In [ ]:
# If you are running on Google Colab, UNCOMMENT this block.
'''
# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
'''

In [ ]:
# =============================================================================
# IMPORTS, PATHS, CONSTANTS
# =============================================================================

from pathlib import Path
from typing import Dict, List, Tuple, Any

import os
import sys
import json
import time
import random
import warnings
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter

from scipy.stats import mannwhitneyu
from tqdm import tqdm, trange

import torch

warnings.filterwarnings("ignore")
torch.set_grad_enabled(False)

# =============================================================================
# EXPERIMENT SETTINGS
# =============================================================================

MASTER_SEED = 42

# for faster run NUM_RUNS=1, EPISODE_BUDGET= 100. for paper results replication, set (NUM_RUNS=10, EPISODE_BUDGET= 1500)

NUM_RUNS = 10

EPISODE_BUDGET = 1500

BASE_SEED_START = 5000

ROWS = 6
COLS = 7

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
torch.manual_seed(MASTER_SEED)

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/mbrl-testing-frameworks-empirical-study"
)

AGENT_ROOT = (
    ROOT
    / "agents"
    / "muzero"
    / "connect4"
)

RESULTS_DIR = (
    ROOT
    / "results"
    / "connect4"
)

RAW_DIR = RESULTS_DIR / "raw"
TABLES_DIR = RESULTS_DIR / "tables"
FIGS_DIR = RESULTS_DIR / "figs"

for p in [
    RESULTS_DIR,
    RAW_DIR,
    TABLES_DIR,
    FIGS_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Device :", DEVICE)
print("Agent  :", AGENT_ROOT)
print("Results:", RESULTS_DIR)

In [ ]:
# =============================================================================
# MUZERO AGENT LOADING
# =============================================================================

LOCAL_MUZERO = Path(
    "external/muzero-general"
)

if not LOCAL_MUZERO.exists():

    os.system(
        "git clone --depth=1 "
        "https://github.com/werner-duvaud/muzero-general.git "
        "external/muzero-general"
    )

sys.path.insert(
    0,
    str(LOCAL_MUZERO),
)

import games.connect4 as c4_games

models = importlib.import_module(
    "models"
)


def make_config():

    try:
        return c4_games.MuZeroConfig()

    except TypeError:
        return c4_games.MuZeroConfig


def make_network(cfg):

    for name in [
        "MuZeroNetwork",
        "Network",
        "ResidualNetwork",
        "ResNet",
    ]:

        if hasattr(models, name):

            net = getattr(
                models,
                name,
            )(cfg)

            if hasattr(
                net,
                "initial_inference",
            ):
                return net

    raise RuntimeError(
        "MuZero network not found."
    )


def strip_prefix(
    state_dict,
    prefixes,
):

    for p in prefixes:

        stripped = {
            k[len(p):]: v
            for k, v in state_dict.items()
            if k.startswith(p)
        }

        if stripped:
            return stripped

    return state_dict


COMP_DIR = AGENT_ROOT / "components"

agent_cfg = json.loads(
    (
        COMP_DIR
        / "config.json"
    ).read_text()
)

NET_CFG = make_config()

NET_CFG.channels = agent_cfg["channels"]
NET_CFG.blocks = agent_cfg["blocks"]

NET_CFG.action_space_size = COLS

NET_CFG.observation_shape = (
    3,
    ROWS,
    COLS,
)

NET = make_network(NET_CFG)

NET.representation_network.load_state_dict(
    strip_prefix(
        torch.load(
            COMP_DIR / "representation.pt",
            map_location="cpu",
        ),
        [
            "representation_network.",
            "model.representation_network.",
        ],
    ),
    strict=False,
)

NET.dynamics_network.load_state_dict(
    strip_prefix(
        torch.load(
            COMP_DIR / "dynamics.pt",
            map_location="cpu",
        ),
        [
            "dynamics_network.",
            "model.dynamics_network.",
        ],
    ),
    strict=False,
)

NET.prediction_network.load_state_dict(
    strip_prefix(
        torch.load(
            COMP_DIR / "prediction.pt",
            map_location="cpu",
        ),
        [
            "prediction_network.",
            "model.prediction_network.",
        ],
    ),
    strict=False,
)

NET.to(DEVICE)
NET.eval()

print("✓ MuZero loaded")


# ==============================================================================
# SECTION 6 — MUZERO INFERENCE WRAPPERS
# ==============================================================================
#
# Two inference calls used throughout:
#   initial_inference(obs)      → hidden state h₀, policy logits, value
#   recurrent_inference(h, a)   → next hidden state h', reward, policy, value
#
# All inference is done under torch.inference_mode() (no gradients).

def _ensure_4d(t: torch.Tensor) -> torch.Tensor:
    if not torch.is_tensor(t):
        t = torch.tensor(np.array(t), dtype=torch.float32, device=DEVICE)
    else:
        t = t.to(DEVICE)

    if   t.dim() == 3: t = t.unsqueeze(0)
    elif t.dim() == 2: t = t.unsqueeze(-1).unsqueeze(-1)
    elif t.dim() == 1: t = t.view(1, -1, 1, 1)

    return t.contiguous()


def _to_device_obs(obs_np: np.ndarray) -> torch.Tensor:
    return torch.tensor(obs_np[None, ...], dtype=torch.float32, device=DEVICE)


def _extract_output(out):
    hidden = getattr(out, "hidden_state", None)
    policy_logits = getattr(out, "policy_logits", None)
    value = getattr(out, "value", None)
    reward = getattr(out, "reward", None)

    if isinstance(out, tuple):
        # Common MuZero order variants
        if len(out) >= 3:
            value, reward, policy_logits, hidden = out[:4] if len(out) >= 4 else (*out[:3], None)

    return hidden, policy_logits, value, reward


@torch.inference_mode()
def muzero_initial_inference(obs_np):

    obs_t = _to_device_obs(obs_np)

    out = NET.initial_inference(obs_t)

    hidden, pol, val, _ = _extract_output(out)

    if hidden is None:

        if hasattr(
            NET,
            "representation_network",
        ):
            hidden = NET.representation_network(obs_t)
        else:
            raise RuntimeError(
                "Cannot extract hidden state."
            )

    if pol is None:

        if hasattr(
            NET,
            "prediction_network",
        ):

            pred = NET.prediction_network(hidden)

            _, pol, val, _ = _extract_output(pred)

        else:

            pol_np = np.zeros(
                COLS,
                dtype=np.float32,
            )

            return {
                "hidden": hidden,
                "policy_logits": pol_np,
                "value": 0.0,
            }

    pol_np = (
        pol.detach()
        .cpu()
        .numpy()
        .reshape(-1)[:COLS]
    )

    val_f = (
        float(
            val.detach()
            .cpu()
            .numpy()
            .mean()
        )
        if torch.is_tensor(val)
        else 0.0
    )

    return {
        "hidden": hidden.detach(),
        "policy_logits": pol_np,
        "value": val_f,
    }

@torch.inference_mode()
def muzero_recurrent_inference(
    hidden,
    action,
):

    h = hidden.to(DEVICE)

    action_t = torch.tensor(
        [[int(action)]],
        dtype=torch.long,
        device=DEVICE,
    )

    out = NET.recurrent_inference(
        h,
        action_t,
    )

    next_h, pol, val, rew = _extract_output(out)

    return {
        "hidden_next": next_h.detach(),
        "policy_logits":
            pol.detach()
            .cpu()
            .numpy()
            .reshape(-1)[:COLS],
        "value":
            float(
                val.detach()
                .cpu()
                .numpy()
                .mean()
            ),
        "reward":
            float(
                rew.detach()
                .cpu()
                .numpy()
                .mean()
            )
            if torch.is_tensor(rew)
            else 0.0,
    }

In [ ]:
# =============================================================================
# CONNECT4 ENVIRONMENT
# =============================================================================

class C4:

    __slots__ = (
        "board",
        "player",
        "done",
        "winner",
        "moves",
    )

    def __init__(self):

        self.board = np.zeros(
            (ROWS, COLS),
            dtype=np.int8,
        )

        self.player = 1
        self.done = False
        self.winner = 0
        self.moves = 0

    def clone(self):

        c = C4()

        c.board = self.board.copy()
        c.player = self.player
        c.done = self.done
        c.winner = self.winner
        c.moves = self.moves

        return c

    def legal_actions(self):

        return [
            c
            for c in range(COLS)
            if self.board[0, c] == 0
        ]

    def step(self, action):

        if self.done:
            return

        if action not in self.legal_actions():

            self.done = True
            self.winner = -self.player
            return

        row = ROWS - 1

        while (
            row >= 0
            and self.board[row, action] != 0
        ):
            row -= 1

        self.board[row, action] = self.player

        self.moves += 1

        if self._is_win(row, action):

            self.done = True
            self.winner = self.player

        elif self.moves >= ROWS * COLS:

            self.done = True
            self.winner = 0

        else:

            self.player *= -1

    def _is_win(self, row, col):

        pl = self.player

        for dr, dc in [
            (1,0),
            (0,1),
            (1,1),
            (1,-1),
        ]:

            cnt = 1

            rr, cc = row + dr, col + dc

            while (
                0 <= rr < ROWS
                and 0 <= cc < COLS
                and self.board[rr,cc] == pl
            ):
                cnt += 1
                rr += dr
                cc += dc

            rr, cc = row - dr, col - dc

            while (
                0 <= rr < ROWS
                and 0 <= cc < COLS
                and self.board[rr,cc] == pl
            ):
                cnt += 1
                rr -= dr
                cc -= dc

            if cnt >= 4:
                return True

        return False

    def to_obs(self):

        ours = (
            self.board == self.player
        ).astype(np.float32)

        theirs = (
            self.board == -self.player
        ).astype(np.float32)

        ones = np.ones_like(
            ours,
            dtype=np.float32,
        )

        return np.stack(
            [ours, theirs, ones],
            axis=0,
        )


# =============================================================================
# ORACLE
# =============================================================================

def evaluate_oracle(g, action):

    if action not in g.legal_actions():
        return 1

    return 0


# =============================================================================
# TACTICAL FAILURE ORACLE
# =============================================================================

def _side_has_immediate_win(g):

    for a in g.legal_actions():

        gc = g.clone()

        gc.step(a)

        if gc.done and gc.winner == g.player:
            return True

    return False


def oracle_missed_immediate_win(
    g,
    action,
):

    legal = g.legal_actions()

    win_exists = False
    chosen_is_win = False

    for cand in legal:

        gc = g.clone()

        gc.step(cand)

        if gc.done and gc.winner == g.player:

            win_exists = True

            if cand == action:
                chosen_is_win = True

    return (
        win_exists
        and not chosen_is_win
    )


def _opponent_has_immediate_win(g):

    opp = g.clone()

    opp.player *= -1

    for a in opp.legal_actions():

        gc = opp.clone()

        gc.step(a)

        if gc.done and gc.winner == opp.player:
            return True

    return False


def oracle_missed_immediate_block(
    g,
    action,
):

    if not _opponent_has_immediate_win(g):
        return False

    blocking_moves = set()

    for a in g.legal_actions():

        after = g.clone()

        after.step(a)

        if after.done:

            blocking_moves.add(a)

            continue

        if not _opponent_has_immediate_win(after):
            blocking_moves.add(a)

    return action not in blocking_moves


def oracle_immediate_blunder(
    g,
    action,
):

    if action not in g.legal_actions():
        return False

    after = g.clone()

    after.step(action)

    if after.done:
        return False

    return _side_has_immediate_win(after)


def evaluate_oracle_detail(
    g,
    action,
):

    illegal = int(
        action not in g.legal_actions()
    )

    if illegal:

        return {
            "illegal": 1,
            "missed_win": 0,
            "missed_block": 0,
            "blunder": 0,
            "any_failure": 1,
        }

    missed_win = int(
        oracle_missed_immediate_win(
            g,
            action,
        )
    )

    missed_block = int(
        oracle_missed_immediate_block(
            g,
            action,
        )
    )

    blunder = int(
        oracle_immediate_blunder(
            g,
            action,
        )
    )

    any_failure = int(
        illegal
        or missed_win
        or missed_block
        or blunder
    )

    return {
        "illegal": illegal,
        "missed_win": missed_win,
        "missed_block": missed_block,
        "blunder": blunder,
        "any_failure": any_failure,
    }


def evaluate_oracle(
    g,
    action,
):

    return evaluate_oracle_detail(
        g,
        action,
    )["any_failure"]

In [ ]:
# =============================================================================
# METRICS
# =============================================================================

def compute_metrics(
    results_df,
    tcp_time=0.0,
):

    failures = results_df["failure"].values.astype(int)
    times = results_df["time"].values.astype(float)

    n = len(failures)

    total_failures = int(np.sum(failures))

    FR = total_failures / n if n > 0 else np.nan

    if total_failures > 0:

        first_idx = int(
            np.where(failures == 1)[0][0]
        )

        TTF_tests = first_idx + 1

        TTF_time = float(
            np.sum(
                times[: first_idx + 1]
            )
        )

    else:

        TTF_tests = np.nan
        TTF_time = np.nan

    if total_failures > 0:

        positions = (
            np.where(failures == 1)[0]
            + 1
        )

        APFD = (
            1.0
            - (
                np.sum(positions)
                / (n * total_failures)
            )
            + (
                1.0
                / (2.0 * n)
            )
        )

    else:

        APFD = np.nan

    if total_failures > 0:

        cumulative_time = np.cumsum(times)

        fail_times = cumulative_time[
            np.where(failures == 1)[0]
        ]

        total_time = cumulative_time[-1]

        APFD_time = (
            1.0
            - (
                np.sum(fail_times)
                / (
                    total_time
                    * total_failures
                )
            )
            + (
                1.0
                / (2.0 * total_time)
            )
        )

    else:

        APFD_time = np.nan

    return {
        "FR": float(FR),
        "TTF_tests": float(TTF_tests),
        "TTF_time": float(TTF_time),
        "APFD": float(APFD),
        "APFD_time": float(APFD_time),
        "TCP_time": float(tcp_time),
        "Per_test_overhead": float(
            tcp_time / n
        ) if n > 0 else np.nan,
    }

# =============================================================================
# MUZERO SIGNALS
# =============================================================================

def softmax_np(logits):

    x = logits.astype(np.float64)

    x = x - np.max(x)

    p = np.exp(x)

    return p / np.sum(p)
# =============================================================================
# SIGNAL EXTRACTION HELPERS
# =============================================================================

def extract_latent_states(ep):

    if "latent_history" in ep:
        return ep["latent_history"]

    return []


def _policy_entropy(logits):

    probs = softmax_np(logits)

    return float(
        -np.sum(
            probs * np.log(
                probs + 1e-12
            )
        )
    )


def _policy_confidence(logits):

    probs = softmax_np(logits)

    return float(
        np.max(probs)
    )


def compute_comparison_signals(ep):

    entropies = []
    confidences = []

    for obs in ep["board_history"]:

        out = muzero_initial_inference(obs)

        logits = out["policy_logits"]

        entropies.append(
            _policy_entropy(logits)
        )

        confidences.append(
            _policy_confidence(logits)
        )

    return {
        "Entropy":
            float(np.mean(entropies))
            if entropies else 0.0,

        "Confidence":
            float(np.mean(confidences))
            if confidences else 1.0,
    }


def compute_latent_variance(
    latent_history,
):

    if len(latent_history) < 2:
        return 0.0

    H = np.stack(
        latent_history,
        axis=0,
    )

    return float(
        np.mean(
            np.var(
                H,
                axis=0,
            )
        )
    )

def compute_latent_variance(
    latent_history,
):

    if len(latent_history) < 2:
        return 0.0

    H = np.stack(
        latent_history,
        axis=0,
    )

    return float(
        np.mean(
            np.var(
                H,
                axis=0,
            )
        )
    )


# =============================================================================
# TCP METHODS
# =============================================================================

def tcp_fifo(pool):

    return sorted(
        pool,
        key=lambda x:
        x["episode_id"]
    )


def tcp_random(
    pool,
    seed=MASTER_SEED,
):

    rng = np.random.default_rng(seed)

    shuffled = pool.copy()

    rng.shuffle(shuffled)

    return shuffled


def tcp_entropy(pool):

    return sorted(
        pool,
        key=lambda x:
        x["signal_entropy"],
        reverse=True,
    )


def tcp_confidence(pool):

    return sorted(
        pool,
        key=lambda x:
        x["signal_confidence"],
    )


def tcp_latent_variance(pool):

    return sorted(
        pool,
        key=lambda x:
        x["latent_variance"],
        reverse=True,
    )


TCP_REGISTRY = {
    "FIFO": tcp_fifo,
    "Random": tcp_random,
    "Entropy": tcp_entropy,
    "Confidence": tcp_confidence,
    "LatentVariance": tcp_latent_variance,
}


def apply_tcp(
    pool,
    tcp_name,
    run_seed,
):

    func = TCP_REGISTRY[tcp_name]

    start = time.perf_counter()

    if tcp_name == "Random":

        ordered = func(
            pool,
            seed=run_seed,
        )

    else:

        ordered = func(pool)

    tcp_time = (
        time.perf_counter()
        - start
    )

    return ordered, tcp_time

In [ ]:
# =============================================================================
# PHASE 1 — TCP ONLY
# =============================================================================

def choose_action(
    game,
    policy_logits,
):

    logits = policy_logits.copy()

    mask = np.full(
        COLS,
        -np.inf,
    )

    for a in game.legal_actions():
        mask[a] = 0.0

    logits = logits + mask

    return int(
        np.argmax(logits)
    )


def generate_test_pool(
    base_seed,
    budget=EPISODE_BUDGET,
):

    pool = []

    rng = np.random.default_rng(
        base_seed
    )

    for ep_id in tqdm(
        range(budget),
        desc="Generating Pool",
        leave=False,
    ):

        seed = int(
            base_seed + ep_id
        )

        game = C4()

        rollout_moves = int(
            rng.integers(
                0,
                15,
            )
        )

        for _ in range(rollout_moves):

            legal = game.legal_actions()

            if (
                not legal
                or game.done
            ):
                break

            game.step(
                int(
                    rng.choice(legal)
                )
            )

        board_history = []
        action_history = []
        legal_history = []
        latent_history = []

        start = time.perf_counter()

        out = muzero_initial_inference(
            game.to_obs()
        )

        hidden = out["hidden"]
        policy = out["policy_logits"]

        latent_history.append(
            hidden.detach()
            .cpu()
            .numpy()
            .flatten()
        )

        failure = 0

        while not game.done:

            legal = game.legal_actions()

            legal_history.append(
                legal.copy()
            )

            action = choose_action(
                game,
                policy,
            )

            oracle_info = evaluate_oracle_detail(
                game,
                action,
            )

            failure = int(
                failure
                or oracle_info["any_failure"]
            )
            action_history.append(
                action
            )

            game.step(action)

            board_history.append(
                game.to_obs()
            )

            if game.done:
                break

            rec = muzero_recurrent_inference(
                hidden,
                action,
            )

            hidden = rec["hidden_next"]

            policy = rec["policy_logits"]

            latent_history.append(
                hidden.cpu()
                .numpy()
                .flatten()
            )

        wall_time = (
            time.perf_counter()
            - start
        )

        pool.append({
            "episode_id": ep_id,
            "seed": seed,

            "board_history":
                board_history,

            "action_history":
                action_history,

            "legal_actions_history":
                legal_history,

            "latent_history":
                latent_history,

            "failure":
                int(failure),

            "wall_time":
                float(wall_time),

            "steps":
                len(action_history),
        })

    return pool


# =============================================================================
# SIGNAL EXTRACTION
# =============================================================================

def compute_signals_for_pool(pool):

    augmented = []

    for ep in pool:

        ep = dict(ep)

        latent_var = (
            compute_latent_variance(
                ep["latent_history"]
            )
        )

        ep["latent_variance"] = (
            latent_var
        )

        entropies = []
        confidences = []

        for obs in ep["board_history"]:

            out = (
                muzero_initial_inference(
                    obs
                )
            )

            logits = (
                out["policy_logits"]
            )

            probs = softmax_np(
                logits
            )

            entropies.append(
                float(
                    -np.sum(
                        probs
                        * np.log(
                            probs
                            + 1e-12
                        )
                    )
                )
            )

            confidences.append(
                float(
                    np.max(probs)
                )
            )

        ep["signal_entropy"] = (
            float(
                np.mean(entropies)
            )
            if entropies else 0.0
        )

        ep["signal_confidence"] = (
            float(
                np.mean(
                    confidences
                )
            )
            if confidences else 1.0
        )

        augmented.append(ep)

    return augmented


# =============================================================================
# EVALUATION
# =============================================================================

def run_tcp_evaluation(
    ordered_pool,
    tcp_time,
):

    df = pd.DataFrame({
        "failure": [
            ep["failure"]
            for ep in ordered_pool
        ],
        "time": [
            ep["wall_time"]
            for ep in ordered_pool
        ],
    })

    metrics = compute_metrics(
        df,
        tcp_time=tcp_time,
    )

    return df, metrics


# =============================================================================
# PHASE 1
# =============================================================================

def run_phase1_single_run(
    run_id,
    base_seed,
):

    print(
        f"\nPhase 1 | Run {run_id + 1} "
        f"(seed={base_seed})"
    )

    pool = generate_test_pool(
        base_seed,
        EPISODE_BUDGET,
    )

    pool = compute_signals_for_pool(
        pool
    )

    rows = []

    for tcp_name in TCP_REGISTRY:

        ordered_pool, tcp_time = apply_tcp(
            pool,
            tcp_name,
            run_seed=base_seed,
        )

        df, metrics = (
            run_tcp_evaluation(
                ordered_pool,
                tcp_time,
            )
        )

        row = {
            "run_id": run_id,
            "TCP": tcp_name,
        }

        row.update(metrics)

        rows.append(row)

        print(
            f"{tcp_name:<18}"
            f"APFD={metrics['APFD']:.4f}"
        )

    return rows


def run_phase1(
    num_runs=NUM_RUNS,
):

    all_rows = []

    for run_id in range(num_runs):

        base_seed = (
            BASE_SEED_START
            + run_id * 10000
        )

        rows = run_phase1_single_run(
            run_id,
            base_seed,
        )

        all_rows.extend(rows)

    results = pd.DataFrame(
        all_rows
    )

    results.to_csv(
        TABLES_DIR
        / "phase1_results.csv",
        index=False,
    )

    return results


def summarize_phase1(results):

    metrics = [
        "FR",
        "TTF_tests",
        "TTF_time",
        "APFD",
        "APFD_time",
        "TCP_time",
        "Per_test_overhead",
    ]

    agg = (
        results
        .groupby("TCP")[metrics]
        .agg(["mean", "std"])
        .round(6)
    )

    agg.columns = [
        "_".join(c)
        for c in agg.columns
    ]

    agg = agg.reset_index()

    agg.to_csv(
        TABLES_DIR
        / "phase1_aggregate.csv",
        index=False,
    )

    return agg

In [ ]:
# ==============================================================================
# FRAMEWORKS
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier

def _episode_quality(ep):
    return (int(ep["failure"]), int(ep["steps"]))

def _episode_features(ep):

    lat = np.stack(
        ep["latent_history"],
        axis=0,
    )

    lv = compute_latent_variance(
        ep["latent_history"]
    )

    steps = ep["steps"]

    if (
        "board_history" not in ep
        or len(ep["board_history"]) == 0
    ):
        moves0 = 0

    else:

        first_board = ep["board_history"][0]

        moves0 = int(
            np.count_nonzero(
                first_board[0]
                +
                first_board[1]
            )
        )

    return np.array(
        [
            steps,
            moves0,
            lv,
            float(np.mean(lat)),
            float(np.std(lat)),
            float(np.max(lat)),
            float(np.min(lat)),
        ],
        dtype=float,
    )
def _mutate_board(g, rng, min_mut=1, max_mut=4):
    g = g.clone()
    for _ in range(int(rng.integers(min_mut, max_mut + 1))):
        if g.done:
            break
        legal = g.legal_actions()
        if not legal:
            break
        g.step(int(rng.choice(legal)))
    return g

def _run_episode_from_initial(g, ep_id, seed):
    """
    Run one full MuZero episode starting from given board state g.
    Returns full episode dict (same format as generate_test_pool).
    """

    np.random.seed(seed)

    board_history = []
    action_history = []
    legal_actions_history = []
    latent_history = []

    t0 = perf_counter()

    # initial inference
    out = muzero_initial_inference(g.to_obs())
    h = out["hidden"]
    pol = out["policy_logits"]

    board_history.append(g.to_obs())
    latent_history.append(h.detach().cpu().numpy().flatten())

    ep_failure = 0

    while not g.done:
        legal = g.legal_actions()
        legal_actions_history.append(legal.copy())

        action = choose_action(g, pol)
        action_history.append(action)

        fail_flag = evaluate_oracle(g, action)
        ep_failure = 1 if (ep_failure or fail_flag) else 0

        g.step(action)
        board_history.append(g.to_obs())

        if g.done:
            break

        rec = muzero_recurrent_inference(h, action)
        h = rec["hidden_next"]
        pol = rec["policy_logits"]

        latent_history.append(h.detach().cpu().numpy().flatten())

    wall_time = perf_counter() - t0

    return {
        "episode_id": ep_id,
        "seed": seed,
        "board_history": board_history,
        "action_history": action_history,
        "legal_actions_history": legal_actions_history,
        "latent_history": latent_history,
        "failure": int(ep_failure),
        "wall_time": float(wall_time),
        "steps": len(action_history),
    }

def _build_stratified_initial_states(
    n: int,
    base_seed: int,
) -> List[C4]:
    """
    Build n initial board positions with early/mid/late stratification (1/3 each).
    Uses seeded random for reproducibility.
    """
    rng   = random.Random(base_seed)
    pool  : List[C4] = []

    def roll_to_plies(lo: int, hi: int) -> C4:
        g = C4()
        target = rng.randint(lo, hi)
        for _ in range(target):
            ls = g.legal_actions()
            if not ls or g.done:
                break
            g.step(rng.choice(ls))
        return g

    per_stratum = n // 3
    remainder   = n - 2 * per_stratum

    # Early (0–10 plies)
    for _ in range(per_stratum):
        pool.append(roll_to_plies(0, 10))
    # Mid (11–24 plies)
    for _ in range(per_stratum):
        pool.append(roll_to_plies(11, 24))
    # Late (25–42 plies)
    for _ in range(remainder):
        pool.append(roll_to_plies(25, 42))

    rng.shuffle(pool)
    return pool

# ------------------------------------------------------------------------------
# INDAGO (failure-focused sampling)
# ------------------------------------------------------------------------------

def generate_pool_indago(base_seed, budget=EPISODE_BUDGET,
                         candidate_mult=4, train_size=1200):
    """
    Faithful Indago adaptation:
      candidate pool → label subset → train RF failure predictor
      → rank candidates by predicted failure probability → select top budget.
    """
    candidate_size = budget * candidate_mult
    candidates = generate_test_pool(base_seed, candidate_size)

    candidates = [
        ep
        for ep in candidates
        if len(ep["board_history"]) > 0
    ]

    candidate_size = len(candidates)

    rng = np.random.default_rng(base_seed)
    idx = rng.choice(
        len(candidates),
        size=min(train_size, len(candidates)),
        replace=False
    )
    for i in idx:

        if len(candidates[i]["board_history"]) == 0:

            print(
                "EMPTY BOARD HISTORY",
                i,
                candidates[i]["steps"],
                candidates[i]["failure"],
            )
    X_train = np.array([_episode_features(candidates[i]) for i in idx])
    y_train = np.array([int(candidates[i]["failure"]) for i in idx])

    if len(np.unique(y_train)) < 2:
        scores = np.array([_episode_features(ep)[2] for ep in candidates])
    else:
        clf = RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=base_seed,
            n_jobs=-1,
        )
        clf.fit(X_train, y_train)

        X_all = np.array([_episode_features(ep) for ep in candidates])
        scores = clf.predict_proba(X_all)[:, 1]

    selected_idx = np.argsort(-scores)[:budget]
    selected = []

    for rank, i in enumerate(selected_idx):
        ep = candidates[int(i)]
        ep["episode_id"] = rank
        ep["indago_score"] = float(scores[int(i)])
        selected.append(ep)

    return selected


# ------------------------------------------------------------------------------
# STARLA (mutation-based)
# ------------------------------------------------------------------------------

def generate_pool_starla(base_seed, budget=EPISODE_BUDGET,
                         pop_size=100, elite_frac=0.5):
    """
    Faithful STARLA adaptation:
      evolutionary search over initial board states,
      fitness = failure + oracle risk + latent instability.
    """
    rng = np.random.default_rng(base_seed)
    gens = int(np.ceil(budget / pop_size))

    pop = _build_stratified_initial_states(pop_size, base_seed)
    pool = []

    for gen in trange(gens, desc="STARLA evolving", leave=False):
        scored = []

        for g in pop:
            if len(pool) >= budget:
                break

            ep_id = len(pool)
            ep = _run_episode_from_initial(g.clone(), ep_id, base_seed + ep_id)

            lv = compute_latent_variance(ep["latent_history"])
            fitness = (
                100.0 * int(ep["failure"])
                + float(ep["steps"])
                + 10.0 * lv
            )

            ep["starla_fitness"] = float(fitness)
            pool.append(ep)
            scored.append((fitness, g.clone()))

        if len(pool) >= budget:
            break

        scored.sort(key=lambda x: x[0], reverse=True)
        n_elite = max(1, int(pop_size * elite_frac))
        parents = [g for _, g in scored[:n_elite]]

        offspring = []
        while len(offspring) < pop_size - n_elite:
            parent = parents[int(rng.integers(0, len(parents)))]
            child = _mutate_board(parent, rng, min_mut=1, max_mut=3)
            offspring.append(child)

        pop = parents + offspring

    return pool[:budget]

# ------------------------------------------------------------------------------
# QD-TESTING (diversity in latent space)
# ------------------------------------------------------------------------------

def _qd_bd_connect4(ep):
    first_board = ep["board_history"][0]
    moves = int(np.count_nonzero(first_board[0] + first_board[1]))

    lat = np.stack(ep["latent_history"], axis=0)
    lv = compute_latent_variance(ep["latent_history"])

    move_bin = min(5, moves // 7)
    latent_bin = int(min(9, lv * 1000))

    return (move_bin, latent_bin)

def generate_pool_qd(base_seed, budget=EPISODE_BUDGET,
                     init_size=300):
    """
    Faithful QD-Testing adaptation:
      archive cell = behavior descriptor,
      elite replacement by quality,
      mutations from archive elites.
    """
    rng = np.random.default_rng(base_seed)
    archive = {}
    pool = []

    init_states = _build_stratified_initial_states(init_size, base_seed)

    def try_add(g, seed):
        ep = _run_episode_from_initial(g.clone(), len(pool), seed)
        cell = _qd_bd_connect4(ep)
        quality = _episode_quality(ep)

        if cell not in archive or quality > archive[cell]["quality"]:
            archive[cell] = {
                "state": g.clone(),
                "quality": quality,
            }

        ep["qd_cell"] = str(cell)
        ep["qd_quality"] = str(quality)
        pool.append(ep)

    for i, g in enumerate(init_states):
        if len(pool) >= budget:
            break
        try_add(g, base_seed + i)

    while len(pool) < budget:
        if archive:
            elite = list(archive.values())[int(rng.integers(0, len(archive)))]
            parent = elite["state"]
            child = _mutate_board(parent, rng, min_mut=1, max_mut=4)
        else:
            child = _build_stratified_initial_states(1, base_seed + len(pool))[0]

        try_add(child, base_seed + len(pool))

    for i, ep in enumerate(pool):
        ep["episode_id"] = i

    return pool[:budget]

# ------------------------------------------------------------------------------
# RLMUTATION (perturbation-based)
# ------------------------------------------------------------------------------

def _run_episode_mutant_connect4(g, ep_id, seed, mutant):
    rng = np.random.default_rng(seed)
    g = g.clone()

    board_history = []
    action_history = []
    legal_actions_history = []
    latent_history = []

    t0 = perf_counter()

    out = muzero_initial_inference(g.to_obs())
    h = out["hidden"]
    pol = out["policy_logits"]

    board_history.append(g.to_obs())
    latent_history.append(h.detach().cpu().numpy().reshape(-1))

    ep_failure = 0

    while not g.done:
        legal = g.legal_actions()
        legal_actions_history.append(legal.copy())

        action = choose_action(g, pol)

        if mutant == "RandomAction" and rng.random() < 0.15:
            action = int(rng.choice(legal))

        if mutant == "IllegalAction" and rng.random() < 0.05:
            action = COLS + 1

        fail_flag = evaluate_oracle(g, action)
        ep_failure = int(ep_failure or fail_flag)

        action_history.append(action)

        if action not in legal:
            g.done = True
            g.winner = -g.player
            ep_failure = 1
            break

        g.step(action)
        board_history.append(g.to_obs())

        if g.done:
            break

        rec = muzero_recurrent_inference(h, action)
        h = rec["hidden_next"]
        pol = rec["policy_logits"]

        if mutant == "PolicyNoise":
            pol = pol + rng.normal(0, 0.25, size=pol.shape)

        latent_history.append(h.detach().cpu().numpy().reshape(-1))

    return {
        "episode_id": ep_id,
        "seed": seed,
        "mutant": mutant,
        "board_history": board_history,
        "action_history": action_history,
        "legal_actions_history": legal_actions_history,
        "latent_history": latent_history,
        "failure": int(ep_failure),
        "wall_time": float(perf_counter() - t0),
        "steps": len(action_history),
    }

def generate_pool_rlmutation(base_seed, budget=EPISODE_BUDGET):
    """
    Faithful RLMutation adaptation:
      evaluate under stored mutants and preserve mutant-specific outcomes.
    """
    mutants = ["RandomAction", "PolicyNoise", "IllegalAction", "Clean"]
    share = int(np.ceil(budget / len(mutants)))

    base_states = _build_stratified_initial_states(budget, base_seed)
    pool = []

    for k, mutant in enumerate(mutants):
        for i in range(share):
            if len(pool) >= budget:
                break

            idx = k * share + i
            g = base_states[idx % len(base_states)].clone()
            ep = _run_episode_mutant_connect4(
                g,
                ep_id=len(pool),
                seed=base_seed + idx,
                mutant=mutant,
            )
            pool.append(ep)

    rng = np.random.default_rng(base_seed + 999)
    rng.shuffle(pool)

    for i, ep in enumerate(pool):
        ep["episode_id"] = i

    return pool[:budget]

# ------------------------------------------------------------------------------
# MDPFUZZ (random fuzzing)
# ------------------------------------------------------------------------------

def _coverage_cells_connect4(ep):
    cells = set()

    for obs in ep["board_history"]:
        pieces = int(np.count_nonzero(obs[0] + obs[1]))
        center_ours = int(np.sum(obs[0][:, 3]))
        center_theirs = int(np.sum(obs[1][:, 3]))

        cells.add((
            min(6, pieces // 7),
            min(3, center_ours),
            min(3, center_theirs),
        ))

    return cells

def generate_pool_mdpfuzz(base_seed, budget=EPISODE_BUDGET,
                          init_size=300):
    """
    Faithful MDPFuzz adaptation:
      corpus grows if failure or trajectory coverage novelty appears.
    """
    rng = np.random.default_rng(base_seed)

    corpus = _build_stratified_initial_states(init_size, base_seed)
    global_coverage = set()
    pool = []

    for i, g in enumerate(corpus):
        if len(pool) >= budget:
            break

        ep = _run_episode_from_initial(g.clone(), len(pool), base_seed + i)
        cells = _coverage_cells_connect4(ep)
        new_cells = cells - global_coverage

        if ep["failure"] or new_cells:
            global_coverage |= cells

        ep["coverage_gain"] = len(new_cells)
        pool.append(ep)

    while len(pool) < budget:
        parent = corpus[int(rng.integers(0, len(corpus)))]
        child = _mutate_board(parent, rng, min_mut=1, max_mut=5)

        ep = _run_episode_from_initial(
            child.clone(),
            len(pool),
            base_seed + len(pool),
        )

        cells = _coverage_cells_connect4(ep)
        new_cells = cells - global_coverage

        if ep["failure"] or new_cells:
            corpus.append(child.clone())
            global_coverage |= cells

        ep["coverage_gain"] = len(new_cells)
        pool.append(ep)

    return pool[:budget]


# ------------------------------------------------------------------------------
# NR-RL (same pool, perturbation at execution stage)
# ------------------------------------------------------------------------------
def _run_episode_nrrl_connect4(g, ep_id, seed, p_rand=0.10):
    rng = np.random.default_rng(seed)
    g = g.clone()

    board_history = []
    action_history = []
    legal_actions_history = []
    latent_history = []

    t0 = perf_counter()

    out = muzero_initial_inference(g.to_obs())
    h = out["hidden"]
    pol = out["policy_logits"]

    board_history.append(g.to_obs())
    latent_history.append(h.detach().cpu().numpy().reshape(-1))

    ep_failure = 0

    while not g.done:
        legal = g.legal_actions()
        legal_actions_history.append(legal.copy())

        if rng.random() < p_rand:
            action = int(rng.choice(legal))
        else:
            action = choose_action(g, pol)

        fail_flag = evaluate_oracle(g, action)
        ep_failure = int(ep_failure or fail_flag)

        action_history.append(action)
        g.step(action)
        board_history.append(g.to_obs())

        if g.done:
            break

        rec = muzero_recurrent_inference(h, action)
        h = rec["hidden_next"]
        pol = rec["policy_logits"]
        latent_history.append(h.detach().cpu().numpy().reshape(-1))

    return {
        "episode_id": ep_id,
        "seed": seed,
        "board_history": board_history,
        "action_history": action_history,
        "legal_actions_history": legal_actions_history,
        "latent_history": latent_history,
        "failure": int(ep_failure),
        "wall_time": float(perf_counter() - t0),
        "steps": len(action_history),
    }

def generate_pool_nrrl(base_seed, budget=EPISODE_BUDGET):
    """
    NR-RL generation: clean initial states.
    Perturbation applied in NR-RL episode execution.
    """
    states = _build_stratified_initial_states(budget, base_seed)
    pool = []

    for i, g in enumerate(states):
        ep = _run_episode_nrrl_connect4(
            g,
            ep_id=i,
            seed=base_seed + i,
            p_rand=0.10,
        )
        pool.append(ep)

    return pool


# ==============================================================================
# FRAMEWORK REGISTRY (FINAL)
# ==============================================================================

FRAMEWORK_REGISTRY = {
    "Indago": generate_pool_indago,
    "STARLA": generate_pool_starla,
    "QD-Testing": generate_pool_qd,
    "RLMutation": generate_pool_rlmutation,
    "MDPFuzz": generate_pool_mdpfuzz,
    "NR-RL": generate_pool_nrrl,
}


In [ ]:
# =============================================================================
# PHASE 2 — FRAMEWORK-INTEGRATED TCP
# =============================================================================

def run_phase2_single_run(
    run_id,
    base_seed,
):

    print(
        f"\nPhase 2 | Run {run_id + 1} "
        f"(seed={base_seed})"
    )

    rows = []

    for framework_name, generator in FRAMEWORK_REGISTRY.items():

        print(
            f"\n[{framework_name}] Generating test pool..."
        )

        t0 = time.perf_counter()

        pool = generator(
            base_seed=base_seed,
            budget=EPISODE_BUDGET,
        )

        generation_time = (
            time.perf_counter() - t0
        )

        # ------------------------------------------------------
        # Compute TCP signals
        # ------------------------------------------------------

        pool = compute_signals_for_pool(
            pool
        )

        print(
            f"Generated {len(pool)} tests "
            f"(FR={np.mean([e['failure'] for e in pool]):.3f})"
        )

        # ------------------------------------------------------
        # TCP evaluation
        # ------------------------------------------------------

        for tcp_name in TCP_REGISTRY:

            ordered_pool, tcp_time = apply_tcp(
                pool,
                tcp_name,
                run_seed=base_seed,
            )

            df, metrics = run_tcp_evaluation(
                ordered_pool,
                tcp_time,
            )

            row = {
                "run_id": run_id,
                "framework": framework_name,
                "tcp": tcp_name,
                "generation_time": generation_time,
            }

            row.update(metrics)

            rows.append(row)

            print(
                f"{tcp_name:<18}"
                f"APFD={metrics['APFD']:.4f}"
            )

    return rows


def run_phase2(
    num_runs=NUM_RUNS,
):

    all_rows = []

    for run_id in range(num_runs):

        base_seed = (
            BASE_SEED_START
            + run_id * 10000
        )

        rows = run_phase2_single_run(
            run_id,
            base_seed,
        )

        all_rows.extend(rows)

        pd.DataFrame(
            all_rows
        ).to_csv(
            RAW_DIR / "phase2_checkpoint.csv",
            index=False,
        )

    results = pd.DataFrame(
        all_rows
    )

    results.to_csv(
        TABLES_DIR / "phase2_results.csv",
        index=False,
    )

    return results


# =============================================================================
# PHASE 2 AGGREGATION
# =============================================================================

def summarize_phase2(results):

    metrics = [
        "FR",
        "TTF_tests",
        "TTF_time",
        "APFD",
        "APFD_time",
        "TCP_time",
        "Per_test_overhead",
    ]

    agg = (
        results
        .groupby(
            ["framework", "tcp"]
        )[metrics]
        .agg(["mean", "std"])
        .round(6)
    )

    agg.columns = [
        "_".join(col)
        for col in agg.columns
    ]

    agg = agg.reset_index()

    agg.to_csv(
        TABLES_DIR / "phase2_aggregate.csv",
        index=False,
    )

    return agg

In [ ]:
# =============================================================================
# STATISTICS
# =============================================================================

def cliffs_delta(x, y):

    nx = len(x)
    ny = len(y)

    gt = sum(
        xi > yj
        for xi in x
        for yj in y
    )

    lt = sum(
        xi < yj
        for xi in x
        for yj in y
    )

    return (gt - lt) / (nx * ny)


def phase1_statistics(results):

    fifo = results[
        results["TCP"] == "FIFO"
    ]["APFD"].values

    rows = []

    for tcp in results["TCP"].unique():

        if tcp == "FIFO":
            continue

        vals = results[
            results["TCP"] == tcp
        ]["APFD"].values

        if len(vals) < 2:
            continue

        _, p = mannwhitneyu(
            vals,
            fifo,
            alternative="two-sided",
        )

        rows.append({
            "TCP": tcp,
            "p_value": p,
            "cliffs_delta": cliffs_delta(
                vals,
                fifo,
            ),
        })

    stat_df = pd.DataFrame(rows)

    stat_df.to_csv(
        TABLES_DIR / "phase1_statistics.csv",
        index=False,
    )

    return stat_df


def phase2_statistics(results):

    rows = []

    for framework in results["framework"].unique():

        subset = results[
            results["framework"] == framework
        ]

        fifo = subset[
            subset["tcp"] == "FIFO"
        ]["APFD"].values

        for tcp in subset["tcp"].unique():

            if tcp == "FIFO":
                continue

            vals = subset[
                subset["tcp"] == tcp
            ]["APFD"].values

            if len(vals) < 2:
                continue

            _, p = mannwhitneyu(
                vals,
                fifo,
                alternative="two-sided",
            )

            rows.append({
                "framework": framework,
                "tcp": tcp,
                "p_value": p,
                "cliffs_delta": cliffs_delta(
                    vals,
                    fifo,
                ),
            })

    stat_df = pd.DataFrame(rows)

    stat_df.to_csv(
        TABLES_DIR / "phase2_statistics.csv",
        index=False,
    )

    return stat_df

In [ ]:
# =============================================================================
# ESSENTIAL PLOTS
# =============================================================================

TCP_COLOR_MAP = {
    "FIFO": "#4D4D4D",
    "Random": "#9E9E9E",
    "Entropy": "#1F77B4",
    "Confidence": "#2CA02C",
    "LatentVariance": "#9467BD",
}

TCP_ORDER = [
    "FIFO",
    "Random",
    "Entropy",
    "Confidence",
    "LatentVariance",
]


def save_plot(name):

    plt.savefig(
        FIGS_DIR / f"{name}.pdf",
        bbox_inches="tight",
    )

    plt.savefig(
        FIGS_DIR / f"{name}.png",
        dpi=300,
        bbox_inches="tight",
    )


def plot_phase1_apfd(agg):

    plt.figure(figsize=(6,3))

    ordered = (
        agg
        .set_index("TCP")
        .reindex(TCP_ORDER)
    )

    plt.bar(
        ordered.index,
        ordered["APFD_mean"],
        color=[
            TCP_COLOR_MAP[t]
            for t in ordered.index
        ],
        edgecolor="black",
        linewidth=0.5,
    )

    plt.ylabel("APFD")
    plt.title("Connect4 Phase 1")

    plt.xticks(
        rotation=35,
        ha="right",
    )

    plt.tight_layout()

    save_plot("phase1_apfd")

    plt.show()


def plot_phase2_apfd(agg):

    frameworks = agg["framework"].unique()

    fig, axes = plt.subplots(
        1,
        len(frameworks),
        figsize=(4 * len(frameworks), 3),
        sharey=True,
    )

    if len(frameworks) == 1:
        axes = [axes]

    for ax, framework in zip(
        axes,
        frameworks,
    ):

        subset = (
            agg[
                agg["framework"] == framework
            ]
            .set_index("tcp")
            .reindex(TCP_ORDER)
        )

        ax.bar(
            subset.index,
            subset["APFD_mean"],
            color=[
                TCP_COLOR_MAP[t]
                for t in subset.index
            ],
            edgecolor="black",
            linewidth=0.5,
        )

        ax.set_title(framework)

        ax.tick_params(
            axis="x",
            rotation=40,
        )

    plt.tight_layout()

    save_plot("phase2_apfd")

    plt.show()


def plot_phase2_overhead(agg):

    frameworks = agg["framework"].unique()

    fig, axes = plt.subplots(
        1,
        len(frameworks),
        figsize=(4 * len(frameworks), 3),
        sharey=True,
    )

    if len(frameworks) == 1:
        axes = [axes]

    for ax, framework in zip(
        axes,
        frameworks,
    ):

        subset = (
            agg[
                agg["framework"] == framework
            ]
            .set_index("tcp")
            .reindex(TCP_ORDER)
        )

        ax.bar(
            subset.index,
            subset["TCP_time_mean"],
            color=[
                TCP_COLOR_MAP[t]
                for t in subset.index
            ],
            edgecolor="black",
            linewidth=0.5,
        )

        ax.set_title(framework)

        ax.tick_params(
            axis="x",
            rotation=40,
        )

    plt.tight_layout()

    save_plot("phase2_overhead")

    plt.show()

In [11]:
# =============================================================================
# MAIN
# =============================================================================

def main():

    print("=" * 70)
    print("CONNECT4 TCP REPLICATION")
    print("=" * 70)

    print("\nRunning Phase 1...")

    phase1_results = run_phase1()

    phase1_agg = summarize_phase1(
        phase1_results
    )

    phase1_stats = phase1_statistics(
        phase1_results
    )

    print("\nRunning Phase 2...")

    phase2_results = run_phase2()

    phase2_agg = summarize_phase2(
        phase2_results
    )

    phase2_stats = phase2_statistics(
        phase2_results
    )

    print("\nGenerating plots...")

    plot_phase1_apfd(
        phase1_agg
    )

    plot_phase2_apfd(
        phase2_agg
    )

    plot_phase2_overhead(
        phase2_agg
    )

    print(
        f"\nTables : {TABLES_DIR}"
    )

    print(
        f"Figures: {FIGS_DIR}"
    )

    return {
        "phase1_results": phase1_results,
        "phase2_results": phase2_results,
        "phase1_statistics": phase1_stats,
        "phase2_statistics": phase2_stats,
    }


if __name__ == "__main__":

    outputs = main()

CONNECT4 TCP REPLICATION

Running Phase 1...

Phase 1 | Run 1 (seed=5000)


FIFO              APFD=0.5041
Random            APFD=0.4971
Entropy           APFD=0.5241
Confidence        APFD=0.5240
LatentVariance    APFD=0.5373

Phase 1 | Run 2 (seed=15000)


FIFO              APFD=0.4997
Random            APFD=0.4988
Entropy           APFD=0.5213
Confidence        APFD=0.5210
LatentVariance    APFD=0.5361

Phase 1 | Run 3 (seed=25000)


FIFO              APFD=0.4992
Random            APFD=0.5027
Entropy           APFD=0.5251
Confidence        APFD=0.5242
LatentVariance    APFD=0.5359

Phase 1 | Run 4 (seed=35000)


FIFO              APFD=0.5003
Random            APFD=0.4999
Entropy           APFD=0.5254
Confidence        APFD=0.5247
LatentVariance    APFD=0.5372

Phase 1 | Run 5 (seed=45000)


...

Generated 1500 tests (FR=1.000)
FIFO              APFD=0.5000
Random            APFD=0.5000
Entropy           APFD=0.5000
Confidence        APFD=0.5000
LatentVariance    APFD=0.5000

[STARLA] Generating test pool...


Generated 1500 tests (FR=0.948)
FIFO              APFD=0.4843
Random            APFD=0.4987
Entropy           APFD=0.4984
Confidence        APFD=0.4972
LatentVariance    APFD=0.5252

[QD-Testing] Generating test pool...
Generated 1500 tests (FR=0.510)
FIFO              APFD=0.5019
Random            APFD=0.5112
Entropy           APFD=0.5343
Confidence        APFD=0.5232
LatentVariance    APFD=0.7164

[RLMutation] Generating test pool...
Generated 1500 tests (FR=0.555)
FIFO              APFD=0.4867
Random            APFD=0.4956
Entropy           APFD=0.5056
Confidence        APFD=0.4950
LatentVariance    APFD=0.7189

[MDPFuzz] Generating test pool...
Generated 1500 tests (FR=0.561)
FIFO              APFD=0.4695
Random            APFD=0.5012
Entropy           APFD=0.4673
Confidence        APFD=0.4595
LatentVariance    APFD=0.7176

[NR-RL] Generating test pool...
Generated 1500 tests (FR=0.554)
FIFO              APFD=0.5080
Random            APFD=0.5041
Entropy           APFD=0.5012
Confid

Generated 1500 tests (FR=1.000)
FIFO              APFD=0.5000
Random            APFD=0.5000
Entropy           APFD=0.5000
Confidence        APFD=0.5000
LatentVariance    APFD=0.5000

[STARLA] Generating test pool...


Generated 1500 tests (FR=0.951)
FIFO              APFD=0.4831
Random            APFD=0.4972
Entropy           APFD=0.4993
Confidence        APFD=0.4995
LatentVariance    APFD=0.5243

[QD-Testing] Generating test pool...
Generated 1500 tests (FR=0.580)
FIFO              APFD=0.4986
Random            APFD=0.5033
Entropy           APFD=0.5679
Confidence        APFD=0.5528
LatentVariance    APFD=0.6887

[RLMutation] Generating test pool...
Generated 1500 tests (FR=0.545)
FIFO              APFD=0.4986
Random            APFD=0.5060
Entropy           APFD=0.4997
Confidence        APFD=0.4870
LatentVariance    APFD=0.7226

[MDPFuzz] Generating test pool...
Generated 1500 tests (FR=0.607)
FIFO              APFD=0.4650
Random            APFD=0.5077
Entropy           APFD=0.4920
Confidence        APFD=0.4901
LatentVariance    APFD=0.6957

[NR-RL] Generating test pool...
Generated 1500 tests (FR=0.542)
FIFO              APFD=0.5053
Random            APFD=0.5043
Entropy           APFD=0.4934
Confid

KeyboardInterrupt: 